Multi-Model Comparison for Pothole Detection in Visually Impaired Pedestrian Navigation
Compares YOLOv8m, YOLOv10m, YOLOv11m, Faster R-CNN, and SSD-VGG16 on a merged pothole dataset

In [1]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT, SAVE_DIR, OUTPUT_DIR accordingly so paths work across environments.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_JUPYTER = not ON_KAGGLE and not ON_COLAB  # local Jupyter

if ON_KAGGLE:
    print("Running on KAGGLE")
    ROOT = "/kaggle/working"
    SAVE_DIR = "/kaggle/working/saved_models"
elif ON_COLAB:
    print("Running on GOOGLE COLAB")
    ROOT = "/content"
    SAVE_DIR = "/content/saved_models"
else:
    print(" Running on LOCAL JUPYTER")
    ROOT = "."
    SAVE_DIR = "./saved_models"

OUTPUT_DIR = os.path.join(ROOT, "comparison_results")
DATA_DIR = os.path.join(ROOT, "data")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"   Output     -> {OUTPUT_DIR}")
print(f"   Data       -> {DATA_DIR}")
print(f"   Models     -> {SAVE_DIR}")


 Running on LOCAL JUPYTER
   Output     -> ./comparison_results
   Data       -> ./data
   Models     -> ./saved_models


In [2]:
# Prints installed package versions for NumPy, Pandas, OpenCV, and PyTorch, and confirms GPU availability and device name.
import torch, numpy as np, pandas as pd, cv2

print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print(f"OpenCV     : {cv2.__version__}")
print(f"PyTorch    : {torch.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device     : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU -- inference will be slower.")


try:
    import ultralytics, seaborn, tqdm, kagglehub, yaml

    print(" ultralytics, seaborn, tqdm, kagglehub, pyyaml -- all good!")
except ImportError as e:
    print(f" Missing package: {e}")
    print("  Run in terminal: pip install ultralytics seaborn tqdm kagglehub pyyaml")


NumPy      : 2.2.6
Pandas     : 2.3.3
OpenCV     : 4.13.0
PyTorch    : 2.10.0+cu128
Device     : cuda
GPU: NVIDIA GeForce RTX 3090
 ultralytics, seaborn, tqdm, kagglehub, pyyaml -- all good!


In [3]:
# Downloads all three Kaggle pothole datasets through kagglehub
import kagglehub


print("Downloading datasets via kagglehub ...")
print("   (First run will open a browser to log in to Kaggle)")
print()

path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
print(f"chitholian     -> {path_1}")

path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
print(f"andrewmvd      -> {path_2}")

path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")
print(f"ashishkumar    -> {path_3}")


DATASET_ROOTS = {
    "chitholian": path_1,
    "andrewmvd": path_2,
    "ashishkumar": path_3,
}

print(f"\nDataset roots: {DATASET_ROOTS}")


   (First run will open a browser to log in to Kaggle)

chitholian     -> /home/vr3/.cache/kagglehub/datasets/chitholian/annotated-potholes-dataset/versions/1
andrewmvd      -> /home/vr3/.cache/kagglehub/datasets/andrewmvd/pothole-detection/versions/1
ashishkumar    -> /home/vr3/.cache/kagglehub/datasets/ashishkumarak/training-setzip/versions/1

Dataset roots: {'chitholian': '/home/vr3/.cache/kagglehub/datasets/chitholian/annotated-potholes-dataset/versions/1', 'andrewmvd': '/home/vr3/.cache/kagglehub/datasets/andrewmvd/pothole-detection/versions/1', 'ashishkumar': '/home/vr3/.cache/kagglehub/datasets/ashishkumarak/training-setzip/versions/1'}


In [4]:
# Loads all shared imports and defines the fixed evaluation config, CLASS_NAMES, CONF_THRESH, IOU_THRESH, IMG_SIZE, and the YOLO weight paths.
import time, json, glob, warnings, xml.etree.ElementTree as ET
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict

warnings.filterwarnings("ignore")


CLASS_NAMES = ["pothole"]
CONF_THRESH = 0.25
IOU_THRESH = 0.45
IMG_SIZE = 640
MAX_IMAGES = 150

# YOLO weights
YOLO_WEIGHTS = {
    "YOLOv8m": "yolov8m.pt",
    "YOLOv10m": "yolov10m.pt",
    "YOLOv11m": "yolo11m.pt",
}

PALETTE = {
    "YOLOv8m": "#00d4ff",
    "YOLOv10m": "#3b82f6",
    "YOLOv11m": "#6366f1",
    "Faster R-CNN": "#f97316",
    "SSD-VGG16": "#ec4899",
}

print("Config loaded")
print(f"   Device      : {DEVICE}")
print(f"   Max images  : {MAX_IMAGES}")
print(f"   Conf thresh : {CONF_THRESH}")

Config loaded
   Device      : cuda
   Max images  : 150
   Conf thresh : 0.25


In [5]:
# Defines the unified annotation loader used for all three sources. Traverses each dataset root recursively, reading XML bounding boxes
from pathlib import Path
import xml.etree.ElementTree as ET


def load_annotated_potholes(root, max_imgs=None):
    root = Path(root)
    records = []
    for img_path in list(root.rglob("*.jpg")) + list(root.rglob("*.png")):
        xml_path = img_path.with_suffix(".xml")
        if not xml_path.exists():
            xml_path = img_path.parent.parent / "annotations" / (img_path.stem + ".xml")
        gt_boxes = []
        if xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                for obj in tree.findall("object"):
                    bb = obj.find("bndbox")
                    gt_boxes.append(
                        {
                            "label": (obj.find("name").text or "pothole").lower(),
                            "xmin": float(bb.find("xmin").text),
                            "ymin": float(bb.find("ymin").text),
                            "xmax": float(bb.find("xmax").text),
                            "ymax": float(bb.find("ymax").text),
                        }
                    )
            except Exception:
                pass
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
        if max_imgs is not None and len(records) >= max_imgs:
            break
    return records


import pandas as pd


def load_ashishkumar_csv(root, max_imgs=None):
    root = Path(root)
    csv_path = root / "train" / "labels.csv"
    img_dir = root / "train" / "images"
    df = pd.read_csv(csv_path)
    grouped = df.groupby("ImageID")

    records = []
    img_paths = sorted(img_dir.glob("*.jpg"))[:max_imgs]
    for img_path in img_paths:
        gt_boxes = []
        if img_path.name in grouped.groups:
            for _, row in grouped.get_group(img_path.name).iterrows():
                gt_boxes.append(
                    {
                        "label": "pothole",
                        "xmin": float(row["XMin"]),
                        "ymin": float(row["YMin"]),
                        "xmax": float(row["XMax"]),
                        "ymax": float(row["YMax"]),
                    }
                )
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


In [6]:
# Loads the chitholian dataset through the unified loader and reports the image count.
records_1 = load_annotated_potholes(DATASET_ROOTS["chitholian"])
print(
    f"chitholian: {len(records_1)} images ({sum(len(r['gt_boxes']) for r in records_1)} gt boxes)"
)

chitholian: 665 images (1740 gt boxes)


In [7]:
# Loads the andrewmvd dataset through the same unified loader and reports the image count.
records_2 = load_annotated_potholes(DATASET_ROOTS["andrewmvd"])
print(
    f"andrewmvd: {len(records_2)} images ({sum(len(r['gt_boxes']) for r in records_2)} gt boxes)"
)

andrewmvd: 665 images (1740 gt boxes)


In [8]:
# Loads the ashishkumarak dataset through the CSV loader and reports the image count.
records_3 = load_ashishkumar_csv(DATASET_ROOTS["ashishkumar"])
print(
    f"ashishkumar: {len(records_3)} images ({sum(len(r['gt_boxes']) for r in records_3)} gt boxes)"
)

ashishkumar: 674 images (1371 gt boxes)


In [9]:
# Tags each record with its source dataset so downstream dedup and reporting can trace which images came from where.
for r in records_1:
    r["source"] = "annotated_dataset"

for r in records_2:
    r["source"] = "voc_dataset"

for r in records_3:
    r["source"] = "csv_dataset"

In [10]:
# Merges all three sources, removes byte-identical duplicates by MD5 hash, and defines the single canonical shuffled 80/20 train/val split, seed 42, reused by every cell below.
records = records_1 + records_2 + records_3

print("Total images before dedup:", len(records))


import numpy as np
from PIL import Image

NORM_SIZE = (64, 64)
DEDUP_THRESHOLD = 1.0


def normalized_pixels(path):
    with Image.open(path) as img:
        return np.asarray(
            img.convert("L").resize(NORM_SIZE, Image.LANCZOS), dtype=np.float32
        ).ravel()


annotated_only = [r for r in records if r["gt_boxes"]]
print("Computing normalized pixel arrays for dedup")
all_arrs = np.stack([normalized_pixels(r["image_path"]) for r in annotated_only])

keep_mask = np.ones(len(annotated_only), dtype=bool)
seen_arrs = []
for i in range(len(annotated_only)):
    if not keep_mask[i]:
        continue
    if seen_arrs:
        diffs = np.abs(np.stack(seen_arrs) - all_arrs[i]).mean(axis=1)
        if diffs.min() < DEDUP_THRESHOLD:
            keep_mask[i] = False
            continue
    seen_arrs.append(all_arrs[i])

n_before = len(records)
records = [r for r, keep in zip(annotated_only, keep_mask) if keep]
n_after = len(records)
print("Total images after dedup:", n_after)
print("Duplicates removed:", n_before - n_after)


import random as _random

_random.seed(42)
_annotated = [r for r in records if r["gt_boxes"]]
_shuffled = _annotated.copy()
_random.shuffle(_shuffled)
_split_idx = int(len(_shuffled) * 0.8)
train_recs = _shuffled[:_split_idx]
val_recs = _shuffled[_split_idx:]
test_recs = val_recs
print(
    f"\n Canonical split: train={len(train_recs)}  val/test={len(val_recs)}  "
    f"(seed=42, shuffled, shared by all cells below)"
)

Total images before dedup: 2004
Computing normalized pixel arrays for dedup
Total images after dedup: 926
Duplicates removed: 1078

 Canonical split: train=740  val/test=186  (seed=42, shuffled, shared by all cells below)


In [11]:
# Switch between loading previously trained weights and running fine-tuning from scratch. SKIP_TRAINING = True skips straight to evaluation.

SKIP_TRAINING = True  # Set True after you've trained once


SAVED_MODELS_KAGGLE = "/kaggle/input/your-saved-models-dataset/saved_models"
SAVED_MODELS_LOCAL = "./saved_models"

if SKIP_TRAINING:
    # Pick the right path
    if ON_KAGGLE and os.path.isdir(SAVED_MODELS_KAGGLE):
        SAVE_DIR = SAVED_MODELS_KAGGLE
    elif os.path.isdir(SAVED_MODELS_LOCAL):
        SAVE_DIR = SAVED_MODELS_LOCAL
    else:
        print("No saved models found! Set SKIP_TRAINING = False to train first.")

    # Check all weights exist
    expected = [
        "YOLOv8m_finetuned.pt",
        "YOLOv10m_finetuned.pt",
        "YOLOv11m_finetuned.pt",
        "faster_rcnn_finetuned.pth",
        "ssd_vgg16_finetuned.pth",
    ]
    missing = [f for f in expected if not os.path.exists(f"{SAVE_DIR}/{f}")]
    if missing:
        print(f"Missing weights: {missing}")
        print("Set SKIP_TRAINING = False and run the fine-tuning cells first.")
    else:
        # Pre-populate finetuned_yolo so run cells work directly
        finetuned_yolo = {
            "YOLOv8m": f"{SAVE_DIR}/YOLOv8m_finetuned.pt",
            "YOLOv10m": f"{SAVE_DIR}/YOLOv10m_finetuned.pt",
            "YOLOv11m": f"{SAVE_DIR}/YOLOv11m_finetuned.pt",
        }
        print("Saved model paths loaded -- skipping training!")
        print("Jump straight to the Run / Evaluate cells below.")
else:
    print("SKIP_TRAINING = False, fine-tuning cells will run below.")
    print("After training, set SKIP_TRAINING = True to skip next time.")


Saved model paths loaded -- skipping training!
Jump straight to the Run / Evaluate cells below.


Fine-tuning for all five models.

In [12]:
# Converts the canonical train/val split into YOLO-format image and label directories and writes the data.yaml used for training.
import shutil, yaml
from pathlib import Path

YOLO_DIR = os.path.join(ROOT, "yolo_dataset")
for split in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{YOLO_DIR}/{split}", exist_ok=True)


def convert_to_yolo(rec_list, split):
    skipped = 0
    for rec in rec_list:
        img = cv2.imread(str(rec["image_path"]))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Copy image
        dst_img = f"{YOLO_DIR}/images/{split}/{rec['image_path'].name}"
        shutil.copy(str(rec["image_path"]), dst_img)

        # Write YOLO label .txt
        dst_lbl = f"{YOLO_DIR}/labels/{split}/{rec['image_path'].stem}.txt"
        with open(dst_lbl, "w") as f:
            for g in rec["gt_boxes"]:
                cx = ((g["xmin"] + g["xmax"]) / 2) / w
                cy = ((g["ymin"] + g["ymax"]) / 2) / h
                bw = (g["xmax"] - g["xmin"]) / w
                bh = (g["ymax"] - g["ymin"]) / h
                f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

    print(f"  {split}: {len(rec_list) - skipped} images written  ({skipped} skipped)")


convert_to_yolo(train_recs, "train")
convert_to_yolo(val_recs, "val")

# Write data.yaml
yaml_config = {
    "path": YOLO_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["pothole"],
}
with open(f"{YOLO_DIR}/data.yaml", "w") as f:
    yaml.dump(yaml_config, f)

print(f"\ndata.yaml saved to {YOLO_DIR}/data.yaml")


  train: 740 images written  (0 skipped)
  val: 186 images written  (0 skipped)

data.yaml saved to ./yolo_dataset/data.yaml


In [13]:
# Fine-tunes all three YOLO variants on the converted dataset and stores each model's best weights in finetuned_yolo.
from ultralytics import YOLO

SAVE_DIR = os.path.join(ROOT, "saved_models")
DATA_YAML = f"{YOLO_DIR}/data.yaml"
os.makedirs(SAVE_DIR, exist_ok=True)

finetuned_yolo = {}  # stores path to each model's weights


for key, wt in YOLO_WEIGHTS.items():
    dest = f"{SAVE_DIR}/{key}_finetuned.pt"

    if os.path.exists(dest):
        print(f"{key}: weights already exist -> {dest}  (skipping training)")
        finetuned_yolo[key] = dest
        continue

    if SKIP_TRAINING:
        raise FileNotFoundError(
            f"{key}: SKIP_TRAINING is True but {dest} does not exist.\n"
            f"   Run this notebook where the fine-tuned weights live, or "
            f"set SKIP_TRAINING = False to train from scratch."
        )

    print(f"\n Fine-tuning {key} ...")
    model = YOLO(wt)
    model.train(
        data=DATA_YAML,
        epochs=50,
        imgsz=640,
        batch=16,
        name=f"{key}_pothole",
        exist_ok=True,
        device=DEVICE,
        verbose=False,
    )

    run_dir = f"runs/detect/{key}_pothole/weights"
    src = next(
        (p for p in (f"{run_dir}/best.pt", f"{run_dir}/last.pt") if os.path.exists(p)),
        None,
    )
    if src is None:
        raise FileNotFoundError(
            f"{key}: training produced no weights in {run_dir}.\n"
            f"   The real failure is in the training output above this is "
            f"not a copy problem."
        )
    shutil.copy(src, dest)
    finetuned_yolo[key] = dest
    print(f"{key} saved to {dest}  (from {os.path.basename(src)})")

print("\nAll YOLO models ready!")


YOLOv8m: weights already exist -> ./saved_models/YOLOv8m_finetuned.pt  (skipping training)
YOLOv10m: weights already exist -> ./saved_models/YOLOv10m_finetuned.pt  (skipping training)
YOLOv11m: weights already exist -> ./saved_models/YOLOv11m_finetuned.pt  (skipping training)

All YOLO models ready!


In [14]:
# Sets up the torchvision Dataset and DataLoader and fine-tunes Faster R-CNN with a replaced box predictor head for the pothole class.
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
    ssd300_vgg16,
    SSD300_VGG16_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssd import SSDClassificationHead
from torchvision.models.detection._utils import retrieve_out_channels


class PotholeDataset(Dataset):
    def __init__(self, records, img_size=None):
        self.records = [r for r in records if r["gt_boxes"]]
        self.img_size = img_size

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = cv2.imread(str(rec["image_path"]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        sx, sy = 1.0, 1.0
        if self.img_size:
            img = cv2.resize(img, self.img_size)
            sx = self.img_size[0] / w
            sy = self.img_size[1] / h

        img_tensor = T.ToTensor()(img)

        boxes, labels = [], []
        for g in rec["gt_boxes"]:
            x1 = max(0, g["xmin"] * sx)
            y1 = max(0, g["ymin"] * sy)
            x2 = min(img.shape[1], g["xmax"] * sx)
            y2 = min(img.shape[0], g["ymax"] * sy)
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(1)  # 1 = pothole

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
        }
        return img_tensor, target


def collate_fn(x):
    return tuple(zip(*x))


def train_torch_model(model, dataloader, epochs, name):
    optimizer = torch.optim.SGD(
        model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005
    )
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, targets in dataloader:
            imgs = [img.to(DEVICE) for img in imgs]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss = sum(model(imgs, targets).values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(
            f"  [{name}] Epoch {epoch + 1}/{epochs}  Loss: {total_loss / len(dataloader):.4f}"
        )
    return model


FRCNN_CKPT = f"{SAVE_DIR}/faster_rcnn_finetuned.pth"
if os.path.exists(FRCNN_CKPT):
    print(f"Faster R-CNN weights already exist at {FRCNN_CKPT} , skipping training")
elif SKIP_TRAINING:
    raise FileNotFoundError(
        f"SKIP_TRAINING is True but {FRCNN_CKPT} does not exist.\n"
        f"   Run this notebook where the fine-tuned weights live, or "
        f"set SKIP_TRAINING = False to train from scratch."
    )
else:
    print(" Fine-tuning Faster R-CNN ...")
    frcnn_ft = fasterrcnn_resnet50_fpn_v2(
        weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    )
    in_feat = frcnn_ft.roi_heads.box_predictor.cls_score.in_features
    frcnn_ft.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes=2)
    frcnn_ft = frcnn_ft.to(DEVICE)

    dl_frcnn = DataLoader(
        PotholeDataset(train_recs),
        batch_size=4,
        shuffle=True,
        collate_fn=lambda b: tuple(zip(*b)),
    )
    frcnn_ft = train_torch_model(frcnn_ft, dl_frcnn, epochs=20, name="Faster R-CNN")
    torch.save(frcnn_ft.state_dict(), FRCNN_CKPT)
    print("Faster R-CNN saved!\n")

Faster R-CNN weights already exist at ./saved_models/faster_rcnn_finetuned.pth , skipping training


In [15]:
# Frees GPU memory before fine-tuning SSD-VGG16, then trains SSD with a replaced classification head.
import gc

gc.collect()
torch.cuda.empty_cache()


SSD_CKPT = f"{SAVE_DIR}/ssd_vgg16_finetuned.pth"
_train_ssd_needed = not os.path.exists(SSD_CKPT)
if not _train_ssd_needed:
    print(f"SSD-VGG16 weights already exist -> {SSD_CKPT}  (skipping training)")
elif SKIP_TRAINING:
    raise FileNotFoundError(
        f"SKIP_TRAINING is True but {SSD_CKPT} does not exist.\n"
        f"   Run this notebook where the fine-tuned weights live, or "
        f"set SKIP_TRAINING = False to train from scratch."
    )

if _train_ssd_needed:
    print("Fine-tuning SSD-VGG16 ...")
    ssd_ft = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)
    in_ch = retrieve_out_channels(ssd_ft.backbone, (300, 300))
    num_anchors = ssd_ft.anchor_generator.num_anchors_per_location()
    ssd_ft.head.classification_head = SSDClassificationHead(
        in_ch, num_anchors, num_classes=2
    )
    ssd_ft = ssd_ft.to(DEVICE)


def train_ssd(model, dataloader, epochs):

    optimizer = torch.optim.SGD(
        model.parameters(), lr=0.0005, momentum=0.9, weight_decay=0.0005
    )
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, targets in dataloader:
            imgs = [img.to(DEVICE) for img in imgs]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss = sum(model(imgs, targets).values())

            # skip bad batches
            if torch.isnan(loss) or torch.isinf(loss):
                del imgs, targets, loss
                torch.cuda.empty_cache()
                continue

            optimizer.zero_grad()
            loss.backward()
            # gradient clipping prevents explosion
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            del imgs, targets, loss
            torch.cuda.empty_cache()

        print(
            f"  [SSD-VGG16] Epoch {epoch + 1}/{epochs}  Loss: {total_loss / len(dataloader):.4f}"
        )
    return model


if _train_ssd_needed:
    dl_ssd = DataLoader(
        PotholeDataset(train_recs, img_size=(300, 300)),
        batch_size=1,
        shuffle=True,
        collate_fn=lambda b: tuple(zip(*b)),
    )
    ssd_ft = train_ssd(ssd_ft, dl_ssd, epochs=20)
    torch.save(ssd_ft.state_dict(), SSD_CKPT)
    print("SSD-VGG16 saved!")

SSD-VGG16 weights already exist -> ./saved_models/ssd_vgg16_finetuned.pth  (skipping training)


In [16]:
# Defines box_iou, compute_ap, and evaluate, the shared functions used to score every model's predictions against ground truth.
import numpy as np
from collections import defaultdict


def box_iou(b1, b2):
    ix1 = max(b1[0], b2[0])
    iy1 = max(b1[1], b2[1])
    ix2 = min(b1[2], b2[2])
    iy2 = min(b1[3], b2[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    u = a1 + a2 - inter
    return inter / u if u > 0 else 0.0


def compute_ap_voc11(recalls, precisions):
    ap = 0.0
    for thr in np.linspace(0, 1, 11):
        ps = [p for r, p in zip(recalls, precisions) if r >= thr]
        ap += max(ps) if ps else 0.0
    return ap / 11


def compute_ap(recalls, precisions):
    mrec = np.concatenate(([0.0], np.asarray(recalls, dtype=float), [1.0]))
    mpre = np.concatenate(([1.0], np.asarray(precisions, dtype=float), [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))  # precision envelope
    x = np.linspace(0, 1, 101)
    _trapz = getattr(np, "trapezoid", None) or np.trapz  # numpy>=2 rename
    return float(_trapz(np.interp(x, mrec, mpre), x))


def evaluate(all_gt, all_preds, iou_thr=0.5):
    pl = []
    for idx, preds in enumerate(all_preds):
        for p in preds:
            pl.append((idx, p["conf"], [p["xmin"], p["ymin"], p["xmax"], p["ymax"]]))
    pl.sort(key=lambda x: -x[1])

    num_gt = sum(len(g) for g in all_gt)
    matched = defaultdict(set)
    TP, FP = [], []
    for idx, conf, pb in pl:
        gts = all_gt[idx]
        taken = matched[idx]
        best_iou, best_j = 0.0, -1
        for j, g in enumerate(gts):
            if j in taken:
                continue
            iou = box_iou(pb, [g["xmin"], g["ymin"], g["xmax"], g["ymax"]])
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thr and best_j >= 0:
            TP.append(1)
            FP.append(0)
            taken.add(best_j)
        else:
            TP.append(0)
            FP.append(1)

    tp_c = np.cumsum(TP)
    fp_c = np.cumsum(FP)
    recs = (tp_c / num_gt).tolist() if num_gt > 0 else [0.0]
    precs = (tp_c / np.maximum(tp_c + fp_c, 1e-12)).tolist()
    if not pl or num_gt == 0:
        ap_coco = ap_voc = 0.0
    else:
        ap_coco = compute_ap(recs, precs)
        ap_voc = compute_ap_voc11(recs, precs)

    ttp = int(sum(TP))
    tfp = int(sum(FP))
    tfn = num_gt - ttp
    pr = ttp / (ttp + tfp) if (ttp + tfp) > 0 else 0.0
    rc = ttp / (ttp + tfn) if (ttp + tfn) > 0 else 0.0
    f1 = 2 * pr * rc / (pr + rc) if (pr + rc) > 0 else 0.0
    return {
        "mAP@0.5": round(ap_coco, 4),
        "mAP@0.5(VOC11)": round(ap_voc, 4),
        "Precision": round(pr, 4),
        "Recall": round(rc, 4),
        "F1": round(f1, 4),
        "TP": ttp,
        "FP": tfp,
        "FN": tfn,
    }


FPS_WARMUP = 5


def _sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def fps_from_times(times, warmup=FPS_WARMUP):
    """Throughput in images/sec from a list of per-image durations."""
    t = [x for x in times[warmup:] if x > 0]
    if not t:
        t = [x for x in times if x > 0]
    return round(len(t) / sum(t), 1) if t else 0.0


CONF_SWEEP = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]


def threshold_preds(all_preds, conf):
    return [[p for p in preds if p["conf"] >= conf] for preds in all_preds]


def sweep_best_conf(all_gt, all_preds, conf_grid=None, iou_thr=0.5, verbose=True):
    conf_grid = conf_grid if conf_grid is not None else CONF_SWEEP
    rows = []
    for c in conf_grid:
        m = evaluate(all_gt, threshold_preds(all_preds, c), iou_thr=iou_thr)
        rows.append(
            {
                "conf": c,
                **{k: m[k] for k in ("Precision", "Recall", "F1", "TP", "FP", "FN")},
            }
        )
    df_sweep = pd.DataFrame(rows).set_index("conf")
    best_conf = float(df_sweep["F1"].idxmax())
    best_m = evaluate(all_gt, threshold_preds(all_preds, best_conf), iou_thr=iou_thr)
    if verbose:
        print(
            f"     conf sweep -> best F1={best_m['F1']:.4f} at conf={best_conf:.2f} "
            f"(P={best_m['Precision']:.4f}  R={best_m['Recall']:.4f})"
        )
    return best_conf, best_m, df_sweep


def summarize(all_gt, all_preds, fps_times=None, iou_thr=0.5, verbose=True):
    m_map = evaluate(all_gt, all_preds, iou_thr=iou_thr)
    m_def = evaluate(all_gt, threshold_preds(all_preds, CONF_THRESH), iou_thr=iou_thr)
    best_conf, m_best, _ = sweep_best_conf(
        all_gt, all_preds, iou_thr=iou_thr, verbose=verbose
    )
    out = {
        "mAP@0.5": m_map["mAP@0.5"],
        "Precision": m_def["Precision"],
        "Recall": m_def["Recall"],
        "F1": m_def["F1"],
        "TP": m_def["TP"],
        "FP": m_def["FP"],
        "FN": m_def["FN"],
        "best_conf": best_conf,
        "P@best": m_best["Precision"],
        "R@best": m_best["Recall"],
        "F1@best": m_best["F1"],
    }
    if fps_times:
        out["FPS"] = fps_from_times(fps_times)
    return out


WARMUP_ITERS = 10


def warmup(forward, n=WARMUP_ITERS):
    for _ in range(n):
        forward()
    _sync()


def first_image(recs):
    for rec in recs:
        img = cv2.imread(str(rec["image_path"]))
        if img is not None:
            return img
    return None


print("Metric helpers ready (COCO-101 AP, fixed matcher, conf sweep, throughput FPS)")


Metric helpers ready (COCO-101 AP, fixed matcher, conf sweep, throughput FPS)
